# Benchmark dataset overview

Dataset checks and plots using the current `src/eeg_benchmark` API. Data is loaded through MOABB/Braindecode; no local GDF download directory is required.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from eeg_benchmark.datasets import MOABBDatasetWrapper, split_train_test

In [ ]:
dataset = MOABBDatasetWrapper.load("BNCI2014_001")
windows = dataset.create_event_windows()
train_windows, test_windows = split_train_test(windows)
sfreq = float(dataset.dataset.datasets[0].raw.info["sfreq"])

## Dataset summary

In [ ]:
summary = pd.DataFrame(
    [
        {
            "dataset": dataset.dataset_name,
            "channels": len(dataset.channel_names),
            "sampling_hz": sfreq,
            "classes": len(dataset.event_mapping),
            "train_windows": len(train_windows),
            "test_windows": len(test_windows),
        }
    ]
)

display(summary)
display(pd.DataFrame({"channel": dataset.channel_names}))
display(pd.DataFrame([dataset.event_mapping]))

## Class balance

In [ ]:
id_to_event = {value: key for key, value in dataset.event_mapping.items()}
train_counts = Counter(int(train_windows[index][1]) for index in range(len(train_windows)))
test_counts = Counter(int(test_windows[index][1]) for index in range(len(test_windows)))

label_counts = pd.DataFrame(
    [
        {
            "label": label,
            "event": id_to_event[label],
            "train_count": train_counts[label],
            "test_count": test_counts[label],
        }
        for label in sorted(id_to_event)
    ]
)

display(label_counts)

In [ ]:
ax = label_counts.plot(
    x="event",
    y=["train_count", "test_count"],
    kind="bar",
    figsize=(7, 4),
    rot=0,
)
ax.set_title("Class balance")
ax.set_xlabel("event")
ax.set_ylabel("windows")
plt.tight_layout()

## Representative window

In [ ]:
inputs, label, metadata = train_windows[0]
inputs = torch.as_tensor(inputs)

pd.DataFrame(
    [
        {
            "shape": "x".join(str(size) for size in inputs.shape),
            "label": int(label),
            "event": id_to_event[int(label)],
            "mean_uv": float(inputs.mean()),
            "std_uv": float(inputs.std()),
            "min_uv": float(inputs.min()),
            "max_uv": float(inputs.max()),
            "window_metadata": metadata,
        }
    ]
)

In [ ]:
time_s = np.arange(inputs.shape[-1]) / sfreq
channels_to_plot = dataset.channel_names[:5]

fig, ax = plt.subplots(figsize=(12, 4))
for channel_index, channel_name in enumerate(channels_to_plot):
    offset_uv = channel_index * 80
    ax.plot(
        time_s,
        inputs[channel_index].numpy() + offset_uv,
        label=channel_name,
        linewidth=1,
    )

ax.set_title(f"Example train window: {id_to_event[int(label)]}")
ax.set_xlabel("time (s)")
ax.set_ylabel("amplitude + offset (uV)")
ax.legend(loc="upper right", ncols=len(channels_to_plot))
plt.tight_layout()

## Channel amplitudes

In [ ]:
sample_count = min(256, len(train_windows))
sample_inputs = torch.stack(
    [torch.as_tensor(train_windows[index][0]).float() for index in range(sample_count)]
)

channel_stats = pd.DataFrame(
    {
        "channel": dataset.channel_names,
        "min_uv": sample_inputs.amin(dim=(0, 2)).numpy(),
        "max_uv": sample_inputs.amax(dim=(0, 2)).numpy(),
        "mean_uv": sample_inputs.mean(dim=(0, 2)).numpy(),
        "std_uv": sample_inputs.std(dim=(0, 2)).numpy(),
    }
)

display(channel_stats.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(channel_stats))
ax.vlines(x, channel_stats["min_uv"], channel_stats["max_uv"], color="0.35")
ax.scatter(x, channel_stats["mean_uv"], color="tab:blue", label="mean")
ax.set_title(f"Per-channel amplitude range over {sample_count} train windows")
ax.set_xlabel("channel")
ax.set_ylabel("uV")
ax.set_xticks(x)
ax.set_xticklabels(channel_stats["channel"], rotation=90)
ax.legend()
plt.tight_layout()

## Frequency content

In [ ]:
demeaned = sample_inputs - sample_inputs.mean(dim=-1, keepdim=True)
spectrum = torch.fft.rfft(demeaned, dim=-1)
power = spectrum.abs().pow(2).mean(dim=(0, 1))
freqs = torch.fft.rfftfreq(demeaned.shape[-1], d=1 / sfreq)
freq_mask = freqs <= 60

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(freqs[freq_mask].numpy(), power[freq_mask].numpy(), linewidth=1.5)
ax.axvspan(8, 13, alpha=0.15, color="tab:green", label="mu")
ax.axvspan(13, 30, alpha=0.12, color="tab:orange", label="beta")
ax.set_title(f"Average power spectrum over {sample_count} train windows")
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("mean power")
ax.legend()
plt.tight_layout()